# TabPFN for Uplift Modeling with Meta-Learners

This notebook evaluates whether TabPFN can improve uplift modeling performance when used inside meta-learners (S-learner, T-learner, X-learner) compared to traditional models.

**Dataset**: ACIC 2016

**Meta-learners to test**:
- **S-learner**: Single model predicting outcome with treatment as feature
- **T-learner**: Two separate models for treatment and control groups
- **X-learner**: Two-stage approach that imputes counterfactual outcomes and uses propensity-weighted CATE estimates
- **R-learner**
- **DR-Learner**

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as st
import pickle
from collections import defaultdict
from datetime import datetime
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, KFold
from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
import tabpfn
from tabpfn import TabPFNRegressor, TabPFNClassifier
from tabicl import TabICLRegressor, TabICLClassifier
from econml.metalearners import SLearner, TLearner, XLearner
from econml.dml import NonParamDML, CausalForestDML
from econml.dr import DRLearner
import matplotlib.pyplot as plt
import warnings
import torch

print(tabpfn.__version__)

# Detect device
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f"Using device: {device}")

# CausalPFN does not support MPS
causalpfn_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"CausalPFN device: {causalpfn_device}")

# TabICL does not support MPS
tabicl_device = "cpu" if device == "mps" else device
print(f"TabICL device: {tabicl_device}")


# Set random seed for reproducibility
np.random.seed(42)

# Suppress warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", message=".*feature names.*")

import os
os.environ["PYTHONWARNINGS"] = "ignore::UserWarning"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TABPFN_NO_TELEMETRY_PROMPT"] = "1"

## 2. Define Models, Tuning, and Metrics

We define LightGBM hyperparameter tuning functions (using `RandomizedSearchCV`) and evaluation metrics. LightGBM is tuned **once per dataset** — the best hyperparameters are then reused across all meta-learners for that dataset.

In [ ]:
# ----------------------------------------------------
#                       TUNING
# ----------------------------------------------------

# LightGBM hyperparameter search space
LGBM_GRID = {
    'num_leaves':        [15, 31, 63],
    'min_child_samples': [20, 50, 100],
    'learning_rate':     [0.01, 0.05, 0.1],
    'n_estimators':      [200, 500, 1000],
}

N_ITER = 30    # RandomizedSearchCV draws
N_EST  = 1000  # base tree count (overridden by grid)


def tune_lgbm(X, y, classifier=False, stratify=None, n_iter=N_ITER, seed=42):
    """Tune LGBM via RandomizedSearchCV. Returns best_params_ dict.

    - classifier=False  → LGBMRegressor, scored by neg_MSE
    - classifier=True   → LGBMClassifier, scored by neg_log_loss
    - stratify          → use StratifiedKFold on this variable (regression only);
                          if None, falls back to KFold with adaptive n_splits
    """
    X = np.asarray(X)
    if classifier:
        base    = LGBMClassifier(n_estimators=N_EST, random_state=seed, verbose=-1)
        scoring = 'neg_log_loss'
        cv      = list(StratifiedKFold(3, shuffle=True, random_state=seed).split(X, y))
    else:
        base    = LGBMRegressor(n_estimators=N_EST, random_state=seed, verbose=-1)
        scoring = 'neg_mean_squared_error'
        if stratify is not None:
            cv  = list(StratifiedKFold(3, shuffle=True, random_state=seed).split(X, stratify))
        else:
            n_splits = min(3, max(2, len(y) // 20))
            cv  = list(KFold(n_splits, shuffle=True, random_state=seed).split(X))

    return RandomizedSearchCV(
        base, LGBM_GRID, n_iter=n_iter, scoring=scoring,
        cv=cv, n_jobs=-1, random_state=seed,
    ).fit(X, y).best_params_


def make_lgbm_final(seed=42):
    """LGBM wrapped in RandomizedSearchCV for final-stage tuning on pseudo-outcomes."""
    return RandomizedSearchCV(
        LGBMRegressor(n_estimators=N_EST, random_state=seed, verbose=-1),
        LGBM_GRID, n_iter=N_ITER, cv=3, scoring='neg_mean_squared_error',
        n_jobs=-1, random_state=seed,
    )


from econml.utilities import WeightedModelWrapper


# ── Metric functions ─────────────────────────────────────────────────────────
def calculate_pehe(predicted_ite, true_ite):
    return np.sqrt(mean_squared_error(true_ite, predicted_ite))

def calculate_ate_error(predicted_ite, true_ite):
    return np.abs(predicted_ite.mean() - true_ite.mean())


## 3. Data Loading (ACIC 2016)

We load all 10 instances of the ACIC 2016 dataset via causallib and store them for evaluation. Each instance is split 80/20 into train/test (stratified on treatment).

In [ ]:
from collections import defaultdict
from causallib.datasets import load_acic16
from sklearn.model_selection import train_test_split

# Store results for all runs
# Structure: results[meta_learner][base_model][metric] = list of values
all_results = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

n_datasets = 10  # Use all 10 ACIC 2016 instances
processed_datasets = []

print(f"Loading {n_datasets} ACIC 2016 instances via causallib...")

for i in range(1, n_datasets + 1):
    data = load_acic16(instance=i)
    X_full   = data.X.reset_index(drop=True)
    T_full   = data.a.values
    Y_full   = data.y.values
    mu0_full = data.po['0'].values
    mu1_full = data.po['1'].values
    feature_names = X_full.columns.tolist()

    # Stratified 80/20 train/test split (stratified on treatment)
    idx_train, idx_test = train_test_split(
        np.arange(len(T_full)),
        test_size=0.2,
        random_state=i,
        stratify=T_full
    )

    X_train = X_full.iloc[idx_train].reset_index(drop=True)
    X_test  = X_full.iloc[idx_test].reset_index(drop=True)
    T_train = T_full[idx_train]
    T_test  = T_full[idx_test]
    Y_train = Y_full[idx_train]
    Y_test  = Y_full[idx_test]
    mu0_test = mu0_full[idx_test]
    mu1_test = mu1_full[idx_test]
    true_ITE_test = mu1_test - mu0_test

    processed_datasets.append({
        'id':            i,
        'X_train':       X_train,
        'X_test':        X_test,
        'T_train':       T_train,
        'T_test':        T_test,
        'Y_train':       Y_train,
        'Y_test':        Y_test,
        'mu0_train':     mu0_full[idx_train],
        'mu1_train':     mu1_full[idx_train],
        'true_ITE_test': true_ITE_test,
    })

print(f"Loaded {len(processed_datasets)} instances.")
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}, Features: {X_train.shape[1]}")

## 4. Unified Meta-Learner Evaluation

We evaluate all five meta-learners (S, T, X, R, DR) in a single loop over the 10 ACIC replications. For each dataset, we tune LightGBM once for the outcome model (regressor) and once for the propensity model (classifier), then reuse those tuned models across all meta-learners. This avoids redundant tuning and ensures consistency.

In [ ]:
import time

print("Evaluating all meta-learners across 10 ACIC replications...\n")

for dataset in processed_datasets:
    i = dataset['id']
    n = len(processed_datasets)
    print(f"\n{'='*70}")
    print(f" Dataset {i}/{n}")
    print(f"{'='*70}")

    X_train, X_test = dataset['X_train'], dataset['X_test']
    T_train          = dataset['T_train']
    Y_train          = dataset['Y_train']
    true_ITE_test    = dataset['true_ITE_test']

    # ── Arm splits (needed for T and X learner tuning) ───────────────────────
    ctrl = T_train == 0
    trt  = T_train == 1
    X_ctrl, Y_ctrl = X_train[ctrl], Y_train[ctrl]
    X_trt,  Y_trt  = X_train[trt],  Y_train[trt]

    # ── LightGBM hyperparameter tuning ───────────────────────────────────────
    t0 = time.time()
    X_with_T        = np.column_stack([X_train, T_train])
    params_s        = tune_lgbm(X_with_T, Y_train, stratify=T_train)  # S-learner outcome (X+T features)
    params_outcome  = tune_lgbm(X_train,  Y_train, stratify=T_train)  # outcome nuisance (R/DR)
    params_prop     = tune_lgbm(X_train,  T_train, classifier=True)   # propensity model
    params_ctrl     = tune_lgbm(X_ctrl,   Y_ctrl)                     # T/X control arm outcome
    params_trt      = tune_lgbm(X_trt,    Y_trt)                      # T/X treated arm outcome
    print(f"  LightGBM tuning: {time.time() - t0:.1f}s")

    # ── Model configs ────────────────────────────────────────────────────────
    base_model_configs = {
        'LinearRegression': {
            's_model':       LinearRegression(),
            't_models':      (LinearRegression(), LinearRegression()),
            'x_models':      (LinearRegression(), LinearRegression()),
            'x_cate':        (LinearRegression(), LinearRegression()),
            'x_propensity':  LogisticRegression(max_iter=1000, random_state=42),
            'r_model_y':     LinearRegression(),
            'r_model_t':     LogisticRegression(max_iter=1000, random_state=42),
            'r_model_final': LinearRegression(),
            'dr_regression': LinearRegression(),
            'dr_propensity': LogisticRegression(max_iter=1000, random_state=42),
            'dr_final':      LinearRegression(),
        },
        'LightGBM': {
            's_model':       LGBMRegressor(random_state=42, verbose=-1, **params_s),
            't_models':      (LGBMRegressor(random_state=42, verbose=-1, **params_ctrl), LGBMRegressor(random_state=43, verbose=-1, **params_trt)),
            'x_models':      (LGBMRegressor(random_state=42, verbose=-1, **params_ctrl), LGBMRegressor(random_state=43, verbose=-1, **params_trt)),
            'x_cate':        (make_lgbm_final(46), make_lgbm_final(47)),
            'x_propensity':  LGBMClassifier(random_state=44, verbose=-1, **params_prop),
            'r_model_y':     LGBMRegressor(random_state=42, verbose=-1, **params_outcome),
            'r_model_t':     LGBMClassifier(random_state=43, verbose=-1, **params_prop),
            'r_model_final': make_lgbm_final(44),
            'dr_regression': LGBMRegressor(random_state=42, verbose=-1, **params_outcome),
            'dr_propensity': LGBMClassifier(random_state=44, verbose=-1, **params_prop),
            'dr_final':      make_lgbm_final(45),
        },
        'TabPFN': {
            's_model':       TabPFNRegressor(device=device),
            't_models':      (TabPFNRegressor(device=device), TabPFNRegressor(device=device)),
            'x_models':      (TabPFNRegressor(device=device), TabPFNRegressor(device=device)),
            'x_cate':        (TabPFNRegressor(device=device), TabPFNRegressor(device=device)),
            'x_propensity':  TabPFNClassifier(device=device),
            'r_model_y':     TabPFNRegressor(device=device),
            'r_model_t':     TabPFNClassifier(device=device),
            'r_model_final':  make_lgbm_final(44),  # TabPFN not suited for residual-on-residual
            'dr_regression': TabPFNRegressor(device=device),
            'dr_propensity': TabPFNClassifier(device=device),
            'dr_final':      TabPFNRegressor(device=device),
        },
        'TabICL': {
            's_model':       TabICLRegressor(device=tabicl_device, random_state=42, verbose=False),
            't_models':      (TabICLRegressor(device=tabicl_device, random_state=42, verbose=False),
                              TabICLRegressor(device=tabicl_device, random_state=43, verbose=False)),
            'x_models':      (TabICLRegressor(device=tabicl_device, random_state=42, verbose=False),
                              TabICLRegressor(device=tabicl_device, random_state=43, verbose=False)),
            'x_cate':        (TabICLRegressor(device=tabicl_device, random_state=44, verbose=False),
                              TabICLRegressor(device=tabicl_device, random_state=45, verbose=False)),
            'x_propensity':  TabICLClassifier(device=tabicl_device, random_state=42, verbose=False),
            'r_model_y':     TabICLRegressor(device=tabicl_device, random_state=42, verbose=False),
            'r_model_t':     TabICLClassifier(device=tabicl_device, random_state=42, verbose=False),
            'r_model_final': make_lgbm_final(44), # TabICL not suited for residual-on-residual
            'dr_regression': TabICLRegressor(device=tabicl_device, random_state=42, verbose=False),
            'dr_propensity': TabICLClassifier(device=tabicl_device, random_state=42, verbose=False),
            'dr_final':      TabICLRegressor(device=tabicl_device, random_state=42, verbose=False),
        },
    }

    for name, cfg in base_model_configs.items():
        # S-Learner
        t0 = time.time()
        s_learner = SLearner(overall_model=cfg['s_model'])
        s_learner.fit(Y_train, T_train, X=X_train)
        te = s_learner.effect(X_test)
        all_results['S'][name]['pehe'].append(calculate_pehe(te, true_ITE_test))
        all_results['S'][name]['ate_error'].append(calculate_ate_error(te, true_ITE_test))
        print(f"  S-learner  + {name:20s}: {time.time() - t0:.1f}s")

        # T-Learner
        t0 = time.time()
        t_learner = TLearner(models=cfg['t_models'])
        t_learner.fit(Y_train, T_train, X=X_train)
        te = t_learner.effect(X_test)
        all_results['T'][name]['pehe'].append(calculate_pehe(te, true_ITE_test))
        all_results['T'][name]['ate_error'].append(calculate_ate_error(te, true_ITE_test))
        print(f"  T-learner  + {name:20s}: {time.time() - t0:.1f}s")

        # X-Learner
        t0 = time.time()
        x_learner = XLearner(models=cfg['x_models'], cate_models=cfg['x_cate'], propensity_model=cfg['x_propensity'])
        x_learner.fit(Y_train, T_train, X=X_train)
        te = x_learner.effect(X_test)
        all_results['X'][name]['pehe'].append(calculate_pehe(te, true_ITE_test))
        all_results['X'][name]['ate_error'].append(calculate_ate_error(te, true_ITE_test))
        print(f"  X-learner  + {name:20s}: {time.time() - t0:.1f}s")

        # R-Learner (NonParamDML)
        t0 = time.time()
        r_learner = NonParamDML(
            model_y=cfg['r_model_y'], model_t=cfg['r_model_t'],
            model_final=cfg['r_model_final'], discrete_treatment=True
        )
        r_learner.fit(Y_train, T_train, X=X_train)
        te = r_learner.effect(X_test)
        all_results['R'][name]['pehe'].append(calculate_pehe(te, true_ITE_test))
        all_results['R'][name]['ate_error'].append(calculate_ate_error(te, true_ITE_test))
        print(f"  R-learner  + {name:20s}: {time.time() - t0:.1f}s")

        # DR-Learner
        t0 = time.time()
        dr_learner = DRLearner(
            model_regression=cfg['dr_regression'],
            model_propensity=cfg['dr_propensity'],
            model_final=cfg['dr_final'],
            min_propensity=0.05 #PREVENTS ENORMOUS OUTCOME
        )
        dr_learner.fit(Y_train, T_train, X=X_train)
        te = dr_learner.effect(X_test)
        all_results['DR'][name]['pehe'].append(calculate_pehe(te, true_ITE_test))
        all_results['DR'][name]['ate_error'].append(calculate_ate_error(te, true_ITE_test))
        print(f"  DR-learner + {name:20s}: {time.time() - t0:.1f}s")

    # ── Causal Forest (CausalForestDML) ───────────────────────────────────────
    try:
        t0 = time.time()
        cf = CausalForestDML(
            model_y=LGBMRegressor(random_state=42, verbose=-1, **params_outcome),
            model_t=LGBMClassifier(random_state=43, verbose=-1, **params_prop),
            discrete_treatment=True,
            n_estimators=200,
            min_samples_leaf=5,
            random_state=42,
        )
        cf.tune(Y_train, T_train, X=X_train)
        cf.fit(Y_train, T_train, X=X_train)
        te = cf.effect(X_test)
        all_results['CF']['CausalForest']['pehe'].append(calculate_pehe(te, true_ITE_test))
        all_results['CF']['CausalForest']['ate_error'].append(calculate_ate_error(te, true_ITE_test))
        print(f"  CF         + {'CausalForest':20s}: {time.time() - t0:.1f}s")
    except Exception as e:
        print(f"\nCausalForest error on dataset {i}: {e}")


    # --- CausalPFN ---
    try:
        print(f"  [CausalPFN]...", end=" ", flush=True)
        _t0 = time.time()
        cpfn = CATEEstimator(device=causalpfn_device, verbose=False)
        cpfn.fit(np.asarray(X_train, dtype=np.float32),
                 np.asarray(T_train, dtype=np.float32).reshape(-1),
                 np.asarray(Y_train, dtype=np.float32).reshape(-1))
        te = cpfn.estimate_cate(np.asarray(X_test, dtype=np.float32))
        if "torch" in str(type(te)):
            te = te.detach().cpu().numpy()
        te = np.asarray(te, dtype=np.float32).reshape(-1)
        all_results["CausalPFN"]["CausalPFN"]["pehe"].append(calculate_pehe(te, true_ITE_test))
        all_results["CausalPFN"]["CausalPFN"]["ate_error"].append(calculate_ate_error(te, true_ITE_test))
        print(f"done. ({time.time() - _t0:.1f}s)")
    except Exception as exc:
        print(f"ERROR\n  CausalPFN error on dataset {i}: {exc}")

print("\nAll meta-learner evaluations complete.")

## 5. Aggregated Results

We report the Mean and Standard Error of PEHE and ATE Error across the 100 replications.

In [ ]:

# Aggregate results
summary_rows = []

for meta in ['S', 'T', 'X', 'R', 'DR', 'CF', 'CausalPFN']:
    if meta == 'CausalPFN':
        models = ['CausalPFN']
    elif meta == 'CF':
        models = ['CausalForest']
    else:
        base_models = ['LinearRegression', 'LightGBM', 'TabPFN', 'TabICL']
        models = base_models

    for model in models:
        pehes = all_results[meta][model]['pehe']
        ate_errs = all_results[meta][model]['ate_error']

        if not pehes:
            continue

        n = len(pehes)
        summary_rows.append({
            'Meta-Learner': meta,
            'Base Model': model,
            'PEHE Mean': np.mean(pehes),
            'PEHE SE': np.std(pehes) / np.sqrt(n),
            'ATE Error Mean': np.mean(ate_errs),
            'ATE Error SE': np.std(ate_errs) / np.sqrt(n)
        })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

# Export to CSV
csv_path = 'benchmark_results_ACIC.csv'
df_summary.to_csv(csv_path, index=False)
print(f"\nResults exported to {csv_path}")

# Visualization
if not df_summary.empty:
    df_meta = df_summary[~df_summary['Meta-Learner'].isin(['CausalPFN', 'CF'])]
    df_standalone = df_summary[df_summary['Meta-Learner'].isin(['CausalPFN', 'CF'])]

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    metrics = ['PEHE', 'ATE Error']
    for idx, metric in enumerate(metrics):
        ax = axes[idx]

        if not df_meta.empty:
            plot_data = df_meta.pivot(index='Base Model', columns='Meta-Learner', values=f'{metric} Mean')
            plot_err = df_meta.pivot(index='Base Model', columns='Meta-Learner', values=f'{metric} SE')
            plot_data.plot(kind='bar', yerr=plot_err, ax=ax, capsize=4, rot=0, legend=False)

        for _, row in df_standalone.iterrows():
            val = row[f'{metric} Mean']
        
            # Assign colors
            if row['Meta-Learner'] == 'CausalPFN':
                color = 'red'
            elif row['Meta-Learner'] == 'CF':
                color = 'blue'
            else:
                color = 'black'  # fallback (just in case)
        
            ax.axhline(
                y=val,
                linestyle='--',
                linewidth=1.5,
                color=color,
                label=f"{row['Meta-Learner']}-{row['Base Model']} ({val:.3f})"
            )

        ax.set_title(f'{metric} (Mean \u00b1 SE)')
        ax.set_ylabel(metric)
        ax.grid(True, alpha=0.3, axis='y')
        ax.legend(loc='upper right', framealpha=0.9)

    plt.tight_layout()
    plt.savefig("resultsacic_plot.png", dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No results to plot.")


# 6. Leaderboard

In [ ]:
# Create a ranking leaderboard for top 5 performers
print("=" * 80)
print(" " * 25 + "LEADERBOARD - TOP 5 PERFORMERS")
print("=" * 80)
print()

# Rank by PEHE (lower is better)
df_ranked_pehe = df_summary.sort_values('PEHE Mean').reset_index(drop=True)
df_ranked_pehe['Rank'] = df_ranked_pehe.index + 1

# Rank by ATE Error (lower is better)
df_ranked_ate = df_summary.sort_values('ATE Error Mean').reset_index(drop=True)
df_ranked_ate['Rank'] = df_ranked_ate.index + 1

# Display PEHE Rankings
print("PRECISION IN ESTIMATING HETEROGENEOUS EFFECTS (PEHE)")
print("   Lower is Better - Measures Individual Treatment Effect Accuracy")
print("-" * 80)
for idx, row in df_ranked_pehe.head(5).iterrows():
    rank = idx + 1
    model_name = f"{row['Meta-Learner']}-{row['Base Model']}"
    pehe_score = row['PEHE Mean']
    pehe_se = row['PEHE SE']
    
    print(f"{rank}. {model_name:30s}  PEHE: {pehe_score:6.4f} +/- {pehe_se:5.4f}")

print()
print("=" * 80)
print()

# Display ATE Error Rankings
print("AVERAGE TREATMENT EFFECT ESTIMATION (ATE Error)")
print("   Lower is Better - Measures Population-Level Treatment Effect Accuracy")
print("-" * 80)
for idx, row in df_ranked_ate.head(5).iterrows():
    rank = idx + 1
    model_name = f"{row['Meta-Learner']}-{row['Base Model']}"
    ate_score = row['ATE Error Mean']
    ate_se = row['ATE Error SE']
    
    print(f"{rank}. {model_name:30s}  ATE Error: {ate_score:6.4f} +/- {ate_se:5.4f}")

print()
print("=" * 80)
print()

# Overall winner (combining both metrics with equal weights)
# Normalize scores to 0-1 range and combine
pehe_scores = df_summary['PEHE Mean'].values
ate_scores = df_summary['ATE Error Mean'].values

pehe_normalized = (pehe_scores - pehe_scores.min()) / (pehe_scores.max() - pehe_scores.min())
ate_normalized = (ate_scores - ate_scores.min()) / (ate_scores.max() - ate_scores.min())

df_summary['Combined Score'] = (pehe_normalized + ate_normalized) / 2
df_ranked_overall = df_summary.sort_values('Combined Score').reset_index(drop=True)

print("OVERALL CHAMPION (Combined PEHE + ATE Performance)")
print("-" * 80)
winner = df_ranked_overall.iloc[0]
winner_name = f"{winner['Meta-Learner']}-{winner['Base Model']}"
print(f"Winner: {winner_name}")
print(f"   PEHE: {winner['PEHE Mean']:.4f} +/- {winner['PEHE SE']:.4f}")
print(f"   ATE Error: {winner['ATE Error Mean']:.4f} +/- {winner['ATE Error SE']:.4f}")
print()
print("=" * 80)